# Đo lại pass@1 HumanEval / HumanEval+: v9.2 (merged), base và Instruct

EvalPlus + vLLM, greedy, 1 mẫu mỗi bài. Mỗi lần chạy là một cặp (model, kiểu prompt):

| Kiểu prompt | Cờ EvalPlus | Prompt gửi cho model |
|---|---|---|
| `base` | `--force-base-prompt` | completion thuần: signature + docstring, model viết tiếp |
| `chatml` | (không cờ) | chat template của tokenizer (ChatML, kèm system prompt mặc định của Qwen) |

## 1. Cài đặt

In [1]:
!pip install -q uv
!uv venv /kaggle/working/vllm_env --python 3.12
!uv pip install -q --python /kaggle/working/vllm_env/bin/python "evalplus[vllm]"
!/kaggle/working/vllm_env/bin/python -c "import torch, vllm; print('torch', torch.__version__, 'cuda', torch.version.cuda, '| vllm', vllm.__version__)"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 85.7 MB/s eta 0:00:00
Using CPython 3.12.3 interpreter at: /usr/bin/python3.12
Creating virtual environment at: vllm_env
Activate with: source vllm_env/bin/activate
torch 2.14.0+cu130 cuda 13.0 | vllm 0.33.0


## 2. Config

In [2]:
import os
from datetime import datetime
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

MODELS = {
    "v9.2":       "hoangnh39/qwen2.5-coder-7b-opencode-v9.2-merged",
}
RUNS = ["v9.2:chatml"]  
REF = "instruct_chatml"  
TP, DTYPE = 2, "float16"
EVAL = "/kaggle/working/vllm_env/bin/evalplus.evaluate"

RESULTS = "evalplus_results/he_" + datetime.now().strftime("%Y%m%d_%H%M")
os.makedirs(RESULTS, exist_ok=True)
print("Kết quả:", RESULTS)

!nvidia-smi --query-gpu=name,memory.total --format=csv

Kết quả: evalplus_results/he_20261111_0741
name, memory.total [MiB]
Tesla T4, 15360 MiB
Tesla T4, 15360 MiB


## 3. Kiểm tra trước khi chạy

EvalPlus chỉ dùng chat template khi tokenizer của model có `chat_template`; thiếu thì kiểu `chatml` âm thầm chạy giống `base`. Cell này dừng sớm nếu thiếu, và in token dừng của model.

In [3]:
from transformers import AutoTokenizer, GenerationConfig

for tag in sorted({r.split(":")[0] for r in RUNS if r.endswith(":chatml")}):
    model = MODELS[tag]
    tok = AutoTokenizer.from_pretrained(model)
    assert tok.chat_template, f"{model}: tokenizer thiếu chat_template -> lần chạy chatml sẽ chạy như base"
    print(f"{tag}: chat_template OK | generation_config eos = {GenerationConfig.from_pretrained(model).eos_token_id}")

config.json:   0%|          | 0.00/1.38k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/722 [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

chat_template.jinja:   0%|          | 0.00/2.51k [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/139 [00:00<?, ?B/s]

v9.2: chat_template OK | generation_config eos = [151645, 151643]


## 4. Chạy benchmark

Mỗi lần chạy là một tiến trình riêng nên GPU được giải phóng giữa các lần. Log: `evalplus_<tag>_<prompt>_<dataset>.log`; mẫu và kết quả chi tiết: `evalplus_results/<prompt>/<dataset>/`.

In [4]:
import glob

FLAGS = {"base": "--force-base-prompt", "chatml": ""}
for run in RUNS:
    tag, prompt = run.split(":")
    name = f"{tag}_{prompt}"
    if glob.glob(f"{RESULTS}/{name}/humaneval/*_eval_results.json"):
        print(f"===== {name}: đã có kết quả, bỏ qua =====")
        continue
    model, flag = MODELS[tag], FLAGS[prompt]
    print(f"===== {name}: {model} =====", flush=True)
    !{EVAL} --model {model} --dataset humaneval --backend vllm --greedy {flag} --root {RESULTS}/{name} --tp {TP} --dtype {DTYPE} 2>&1 | tee {RESULTS}/{name}.log | tail -n 12

===== v9.2_chatml: hoangnh39/qwen2.5-coder-7b-opencode-v9.2-merged =====
(EngineCore pid=218) INFO 11-11 08:09:33 [multiproc_executor.py:580] [shutdown] Executor: waiting for worker exit count=2
(Worker_TP0 pid=226) INFO 11-11 08:09:33 [multiproc_executor.py:944] Parent process exited, terminating worker queues
(EngineCore pid=218) INFO 11-11 08:09:36 [multiproc_executor.py:587] [shutdown] Executor: all workers exited gracefully
Computing expected output...
Expected outputs computed in 25.28s
Reading samples...
164it [00:00, 1992.54it/s]
100%|██████████| 164/164 [00:20<00:00,  8.08it/s]
humaneval (base tests)
pass@1:	0.896
humaneval+ (base + extra tests)
pass@1:	0.841
